# LeetCode 1927 — Sum Game bằng Python

Link đề: https://leetcode.com/problems/sum-game/ · Độ khó: **Medium** · Chủ đề: Math, String, Greedy, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def sumGame(self, num: str) -> bool:
        ...
```

Lộ trình: **minimax vét cạn trên chuỗi** (chọn vị trí `?` và chữ số) → **minimax có ghi nhớ trên trạng thái nén `(số ? trái, số ? phải, hiệu)`** → **công thức `O(n)`**: Alice thắng ⇔ tổng số `?` **lẻ**, hoặc `2·(tổng trái − tổng phải) ≠ 9·(số ? phải − số ? trái)`.

## 1. Mục tiêu học

1. Chơi trò chơi tối ưu bằng **minimax**: Alice muốn hai nửa **khác** tổng, Bob muốn **bằng**.
2. **Nén trạng thái**: thứ tự vị trí không quan trọng — chỉ cần số `?` mỗi nửa và **hiệu** tổng hiện tại.
3. Suy ra công thức bằng lập luận **chiến lược đối xứng** ("bắt chước" / ghép cặp `9`): mỗi cặp nước đi (Alice, Bob) trên cùng một nửa có thể đóng góp đúng `9`.
4. Hiểu vai trò của **tính chẵn lẻ số lượt**: ai đi nước cuối có lợi thế quyết định.
5. Kiểm chứng công thức bằng minimax trên mọi chuỗi nhỏ.

## 2. Đề bài & ví dụ

Alice và Bob lần lượt chơi, **Alice đi trước**. Cho chuỗi `num` **độ dài chẵn** gồm chữ số và ký tự `'?'`. Mỗi lượt, nếu còn ít nhất một `'?'`, người chơi:
- chọn một chỉ số `i` có `num[i] == '?'`;
- thay `num[i]` bằng một chữ số bất kỳ từ `'0'` đến `'9'`.

Trò chơi kết thúc khi hết `'?'`. **Bob thắng** nếu tổng chữ số nửa đầu **bằng** tổng chữ số nửa sau; ngược lại **Alice thắng**. Ví dụ `"243801"` → Bob thắng (`2+4+3 = 8+0+1`); `"243803"` → Alice thắng.

Hai người chơi tối ưu. Trả về `true` nếu Alice thắng, `false` nếu Bob thắng.

**Ví dụ 1:** `num = "5023"` → `false` (không có nước đi; `5+0 = 2+3`).

**Ví dụ 2:** `num = "25??"` → `true` (Alice thay một `?` bằng `9`, Bob không thể cân bằng).

**Ví dụ 3:** `num = "?3295???"` → `false`.

**Ràng buộc:** `2 ≤ num.length ≤ 10⁵`, độ dài chẵn, chỉ gồm chữ số và `'?'`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import product
from typing import List

examples = [("5023", False), ("25??", True), ("?3295???", False)]
for s, want in examples:
    print(repr(s), "-> Alice thắng?", want)

def stats(num):
    h = len(num) // 2
    L, R = num[:h], num[h:]
    return (L.count("?"), sum(int(c) for c in L if c != "?"),
            R.count("?"), sum(int(c) for c in R if c != "?"))

print("(? trái, tổng trái, ? phải, tổng phải) của ví dụ 3:", stats("?3295???"))

'5023' -> Alice thắng? False
'25??' -> Alice thắng? True
'?3295???' -> Alice thắng? False
(? trái, tổng trái, ? phải, tổng phải) của ví dụ 3: (1, 14, 3, 5)


## 3. Kiến thức nền

### 3.1 Minimax hai người, mục tiêu đối nghịch

Định nghĩa `win(trạng thái, lượt)` = Alice thắng không. Lượt Alice: thắng nếu **tồn tại** nước đi dẫn tới trạng thái Alice thắng. Lượt Bob: Alice thắng chỉ khi **mọi** nước của Bob đều dẫn tới trạng thái Alice thắng.

In [2]:
# ví dụ nhỏ: một '?' bên trái, hiệu hiện tại d (trái - phải) = -3, lượt Alice
d = -3
print("Alice có nước để kết thúc với hiệu != 0?", any(d + x != 0 for x in range(10)))
print("Nếu đến lượt Bob: Bob có nước làm hiệu = 0?", any(d + x == 0 for x in range(10)))

Alice có nước để kết thúc với hiệu != 0? True
Nếu đến lượt Bob: Bob có nước làm hiệu = 0? True


### 3.2 Nén trạng thái

Chỉ cần biết: `a` = số `?` còn ở nửa trái, `b` = số `?` ở nửa phải, `d` = (tổng trái − tổng phải) hiện tại, và lượt của ai (suy ra từ số `?` đã điền). Điền `x` vào nửa trái: `d += x`; nửa phải: `d −= x`.

In [3]:
a, sl, b, sr = stats("25??")
print(f"25??: a={a}, b={b}, d = {sl} - {sr} = {sl - sr}")

25??: a=0, b=2, d = 7 - 0 = 7


### 3.3 Tính chẵn lẻ số lượt

Tổng số `?` là `q = a + b`. Alice đi các lượt 1, 3, 5, …; nếu `q` **lẻ** thì Alice đi **nước cuối** — chọn chữ số cuối cùng sao cho tổng **khác** (luôn làm được vì 10 lựa chọn cho ra 10 hiệu khác nhau, chỉ một trong số đó bằng 0).

In [4]:
for q in range(1, 6):
    print(q, "dấu ? -> người đi nước cuối:", "Alice" if q % 2 else "Bob")

1 dấu ? -> người đi nước cuối: Alice
2 dấu ? -> người đi nước cuối: Bob
3 dấu ? -> người đi nước cuối: Alice
4 dấu ? -> người đi nước cuối: Bob
5 dấu ? -> người đi nước cuối: Alice


### 3.4 So sánh không dùng chia

Điều kiện `d = 9·(b − a)/2` viết lại thành `2·d == 9·(b − a)` để tránh phép chia (và số thực).

In [5]:
d, a, b = 9, 1, 3
print(2 * d == 9 * (b - a))

True


## 4. Ý tưởng nền tảng / trực giác

Xét `"25??"`: trái `25` (tổng 7), phải `??`. `d = 7`, `a = 0`, `b = 2`. Bob muốn tổng phải bằng 7 sau khi hai người mỗi người điền một chữ số.
- Alice điền `9` ⇒ tổng phải đã ≥ 9 > 7 ⇒ Bob thua. (Hoặc Alice điền `0`, Bob cần `7` — được! Nên Alice phải chọn khôn.)

Xét `"??"` (trái `?`, phải `?`): Alice điền `x` một bên, Bob điền **cùng `x`** bên kia ⇒ bằng nhau ⇒ Bob thắng. Đây là chiến lược **bắt chước**.

Xét trường hợp `?` chỉ ở một nửa, ví dụ phải có 2 dấu `?`, cần tổng phải tăng thêm đúng `d`: Alice điền `x`, Bob điền `9 − x` ⇒ mỗi cặp lượt cộng **đúng 9**. Bob kiểm soát được kết quả **chỉ khi** `d` đúng bằng `9 × (số cặp)`. Nếu `d` khác, Alice đẩy về một phía (chọn `0` hoặc `9`) để Bob không thể bù.

Kết hợp: cặp `?` trái–phải triệt tiêu nhau (bắt chước), phần dư `b − a` dấu `?` ở một nửa tạo `(b − a)/2` cặp, mỗi cặp "bắt buộc" đóng góp 9 ⇒ Bob thắng ⇔ `d = 9·(b − a)/2` (với `q` chẵn).

## 5. Cách 1 — Brute Force / baseline: minimax trên chính chuỗi

**Ý tưởng:** trạng thái là chuỗi hiện tại; mỗi lượt thử **mọi** vị trí `?` và **mọi** chữ số. Ghi nhớ theo chuỗi để chạy được vài `?`.

In [6]:
def sum_game_brute(num):
    h = len(num) // 2

    @lru_cache(maxsize=None)
    def alice_wins(s):
        qs = [i for i, c in enumerate(s) if c == "?"]
        if not qs:
            return sum(map(int, s[:h])) != sum(map(int, s[h:]))
        alice_turn = (num.count("?") - len(qs)) % 2 == 0
        results = (alice_wins(s[:i] + str(x) + s[i + 1:]) for i in qs for x in range(10))
        return any(results) if alice_turn else all(results)

    res = alice_wins(num)
    alice_wins.cache_clear()
    return res

for s, want in examples:
    print(repr(s), sum_game_brute(s), "| kỳ vọng", want)

'5023' False | kỳ vọng False
'25??' True | kỳ vọng True
'?3295???' False | kỳ vọng False


**Độ phức tạp:** số trạng thái chuỗi tới `11^q` (mỗi `?` còn nguyên hoặc là một trong 10 chữ số) ⇒ chỉ chạy với `q ≤ 4`. **Nút thắt:** hai chuỗi có cùng `(a, b, d)` là **cùng một tình huống** — vị trí cụ thể và các chữ số đã điền không quan trọng.

In [7]:
t = time.perf_counter(); r = sum_game_brute("??12??"); dt = time.perf_counter() - t
print(f"'??12??' (4 dấu ?): {r} trong {dt:.2f}s")

'??12??' (4 dấu ?): True trong 0.00s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (nén trạng thái):** `win(a, b, d)` với lượt suy ra từ số `?` đã điền. Số trạng thái `O(a · b · 9(a+b))` — nhỏ hơn nhiều.

**Insight 2 (tính chẵn lẻ):** *nếu* tổng số `?` **lẻ** *thì* Alice đi nước cuối và chọn chữ số làm hai tổng khác nhau ⇒ **Alice thắng**.

**Insight 3 (chiến lược khi `q` chẵn):** đặt `d = sL − sR` và `k = (b − a) / 2` (có thể âm; `b − a` chẵn vì `q` chẵn ⇒ `a`, `b` cùng tính chẵn lẻ).
- *Nếu* `2d == 9(b − a)` *thì* **Bob thắng**: mỗi khi Alice điền `x` vào nửa có nhiều `?` hơn, Bob điền `9 − x` vào **cùng nửa**; khi Alice điền vào nửa kia (nơi cặp trái–phải còn "dư"), Bob bắt chước bên đối diện với **cùng** chữ số. Hiệu cuối cùng = `d − 9·k = 0`.
- *Ngược lại* **Alice thắng**: nếu `d > 9k` Alice luôn điền `9` vào trái / `0` vào phải (đẩy `d` lên), nếu `d < 9k` thì ngược lại; Bob không bù kịp.

**Quy tắc cuối:** `Alice thắng ⇔ (a + b) lẻ  hoặc  2·(sL − sR) ≠ 9·(b − a)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Minimax có ghi nhớ trên trạng thái nén `(a, b, d)`

In [8]:
@lru_cache(maxsize=None)
def win_compressed(a, b, d, alice_turn):
    if a == 0 and b == 0:
        return d != 0
    moves = []
    if a: moves += [(a - 1, b, d + x) for x in range(10)]
    if b: moves += [(a, b - 1, d - x) for x in range(10)]
    if alice_turn:
        return any(win_compressed(na, nb, nd, False) for na, nb, nd in moves)
    return all(win_compressed(na, nb, nd, True) for na, nb, nd in moves)

class SolutionMemo:
    def sumGame(self, num: str) -> bool:
        a, sl, b, sr = stats(num)
        return win_compressed(a, b, sl - sr, True)

for s, want in examples:
    print(SolutionMemo().sumGame(s), want)

False False
True True
False False


**Độ phức tạp:** `d` nằm trong khoảng `[−9(a+b) − 9n, …]`; số trạng thái `O(a · b · 9(a + b))`, mỗi trạng thái 20 nước. Với vài chục `?` vẫn nhanh, nhưng `10⁵` dấu `?` thì không.

### 7.2 Cách 3 — Công thức `O(n)` — lời giải nộp

In [9]:
def sum_game_formula_demo(num):
    a, sl, b, sr = stats(num)
    print(f"? trái a={a}, tổng trái sL={sl}, ? phải b={b}, tổng phải sR={sr}")
    if (a + b) % 2:
        print("tổng số ? lẻ -> Alice đi nước cuối -> Alice thắng"); return True
    lhs, rhs = 2 * (sl - sr), 9 * (b - a)
    print(f"2(sL - sR) = {lhs}, 9(b - a) = {rhs} -> {'bằng: Bob thắng' if lhs == rhs else 'khác: Alice thắng'}")
    return lhs != rhs

for s, want in examples:
    print("=>", sum_game_formula_demo(s), "| kỳ vọng", want); print()

? trái a=0, tổng trái sL=5, ? phải b=0, tổng phải sR=5
2(sL - sR) = 0, 9(b - a) = 0 -> bằng: Bob thắng
=> False | kỳ vọng False

? trái a=0, tổng trái sL=7, ? phải b=2, tổng phải sR=0
2(sL - sR) = 14, 9(b - a) = 18 -> khác: Alice thắng
=> True | kỳ vọng True

? trái a=1, tổng trái sL=14, ? phải b=3, tổng phải sR=5
2(sL - sR) = 18, 9(b - a) = 18 -> bằng: Bob thắng
=> False | kỳ vọng False



In [10]:
class SolutionFormula:
    def sumGame(self, num: str) -> bool:
        h = len(num) // 2
        a = b = sl = sr = 0
        for i, c in enumerate(num):
            if i < h:
                if c == "?": a += 1
                else: sl += ord(c) - 48
            else:
                if c == "?": b += 1
                else: sr += ord(c) - 48
        return (a + b) % 2 == 1 or 2 * (sl - sr) != 9 * (b - a)

for s, want in examples:
    print(SolutionFormula().sumGame(s), want)

False False
True True
False False


**Độ phức tạp:** một lượt duyệt `O(n)`, bộ nhớ `O(1)`.

### 7.3 Cách 4 — Mô phỏng hai chiến lược (kiểm chứng công thức bằng "chơi thật")

Cài **chiến lược Bob** (bắt chước + bù 9) và **chiến lược Alice** (đẩy về một phía), cho đấu với đối thủ **ngẫu nhiên**: khi công thức nói Bob thắng, chiến lược Bob phải luôn thắng; khi công thức nói Alice thắng (với `q` chẵn), chiến lược Alice phải luôn thắng.

In [11]:
def bob_strategy(a, b, d, last):
    side, x = last
    # nếu Alice vừa đi ở nửa còn "nhiều ?" hơn nửa kia (theo số ? dư), Bob bù 9 cùng nửa; ngược lại bắt chước nửa đối diện
    if side == "L":
        if a > b: return ("L", 9 - x)              # sau nước của Alice, trái vẫn dư -> bù cùng nửa
        if b > 0: return ("R", x)                   # bắt chước bên phải
        return ("L", 9 - x)
    else:
        if b > a: return ("R", 9 - x)
        if a > 0: return ("L", x)
        return ("R", 9 - x)

def random_player(rng):
    def p(a, b, d, last=None):
        side = rng.choice(["L"] * a + ["R"] * b)
        return side, rng.randint(0, 9)
    return p

def alice_push(a, b, d, last=None):
    k2 = 9 * (b - a)                                # mục tiêu của Bob: 2d == k2
    if 2 * d > k2:                                  # đẩy d lên
        return ("L", 9) if a else ("R", 0)
    return ("R", 9) if b else ("L", 0)

def duel(num, alice, bob):
    a, sl, b, sr = stats(num); d = sl - sr; last = None; turn = 0
    while a + b:
        if turn % 2 == 0: side, x = alice(a, b, d, last)
        else: side, x = bob(a, b, d, last)
        if side == "L": a -= 1; d += x
        else: b -= 1; d -= x
        last = (side, x); turn += 1
    return d != 0

rng = random.Random(1927)
for _ in range(400):
    h = rng.randint(1, 6)
    num = "".join(rng.choice("0123456789??") for _ in range(2 * h))
    a, sl, b, sr = stats(num)
    formula = SolutionFormula().sumGame(num)
    if (a + b) % 2 == 0 and not formula:          # Bob phải thắng mọi ván với chiến lược của mình
        for _ in range(20):
            assert duel(num, random_player(rng), bob_strategy) is False
    if (a + b) % 2 == 0 and formula:              # Alice đẩy về một phía phải thắng mọi ván
        for _ in range(20):
            assert duel(num, alice_push, random_player(rng)) is True
print("chiến lược Bob / Alice khớp công thức trên 400 chuỗi × 20 ván")

chiến lược Bob / Alice khớp công thức trên 400 chuỗi × 20 ván


`duel` là bộ mô phỏng: mỗi người chơi nhận trạng thái `(a, b, d)` và **nước đi vừa rồi** của đối thủ (Bob cần nó để "đáp trả"). Đối thủ ngẫu nhiên chọn nửa và chữ số tùy ý — chiến lược thắng phải thắng **mọi** ván.

## 8. TRACE từng bước

Trace công thức trên ba ví dụ:

| `num` | `a` | `sL` | `b` | `sR` | `a + b` | `2(sL − sR)` | `9(b − a)` | Alice thắng? |
|---|---|---|---|---|---|---|---|---|
| `5023` | 0 | 5 | 0 | 5 | 0 (chẵn) | 0 | 0 | **false** |
| `25??` | 0 | 7 | 2 | 0 | 2 (chẵn) | 14 | 18 | **true** |
| `?3295???` | 1 | 14 | 3 | 5 | 4 (chẵn) | 18 | 18 | **false** |

Trace một ván mô phỏng `"?3295???"` với Alice ngẫu nhiên, Bob theo chiến lược:

In [12]:
def trace_duel(num, alice, bob):
    a, sl, b, sr = stats(num); d = sl - sr; last = None; turn = 0
    print(f"bắt đầu: a={a}, b={b}, d={d}")
    while a + b:
        who = "Alice" if turn % 2 == 0 else "Bob"
        side, x = (alice if turn % 2 == 0 else bob)(a, b, d, last)
        if side == "L": a -= 1; d += x
        else: b -= 1; d -= x
        print(f"  {who} điền {x} vào nửa {'trái' if side == 'L' else 'phải'} -> a={a}, b={b}, d={d}")
        last = (side, x); turn += 1
    print("kết thúc: d =", d, "->", "Alice thắng" if d else "Bob thắng")
    return d != 0

trace_duel("?3295???", random_player(random.Random(5)), bob_strategy)

bắt đầu: a=1, b=3, d=9
  Alice điền 5 vào nửa phải -> a=1, b=2, d=4
  Bob điền 4 vào nửa phải -> a=1, b=1, d=0
  Alice điền 7 vào nửa trái -> a=0, b=1, d=7
  Bob điền 7 vào nửa phải -> a=0, b=0, d=0
kết thúc: d = 0 -> Bob thắng


False

In [13]:
trace_duel("25??", alice_push, random_player(random.Random(6)))

bắt đầu: a=0, b=2, d=7
  Alice điền 9 vào nửa phải -> a=0, b=1, d=-2
  Bob điền 7 vào nửa phải -> a=0, b=0, d=-9
kết thúc: d = -9 -> Alice thắng


True

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def sumGame(self, num: str) -> bool:
        h = len(num) // 2
        a = b = 0          # số '?' ở nửa trái / phải
        diff = 0           # (tổng trái) - (tổng phải)
        for i, c in enumerate(num):
            if c == "?":
                if i < h: a += 1
                else: b += 1
            else:
                diff += int(c) if i < h else -int(c)
        if (a + b) % 2 == 1:
            return True                       # Alice đi nước cuối
        return 2 * diff != 9 * (b - a)        # Bob thắng <=> diff == 9 * (b - a) / 2

for s, want in examples:
    assert Solution().sumGame(s) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Đặt `a, b` là số `?` hai nửa, `d = sL − sR`.

**(1) `a + b` lẻ ⇒ Alice thắng.** Alice đi nước cuối. Trước nước đó, hiệu là `d'` và còn một `?` ở một nửa; mười chữ số cho mười hiệu cuối khác nhau (`d' ± x`), nhiều nhất một trong số đó bằng 0 ⇒ Alice chọn chữ số khác.

**(2) `a + b` chẵn và `2d = 9(b − a)` ⇒ Bob thắng.** Không mất tổng quát `b ≥ a`, đặt `k = (b − a)/2 ≥ 0`, `d = 9k`. Chiến lược Bob (trả lời ngay sau mỗi nước của Alice): duy trì bất biến "**trước lượt Alice**, hiệu `d = 9·(b − a)/2`". Nếu Alice điền `x` vào nửa phải khi `b > a` ⇒ Bob điền `9 − x` cũng vào phải: `d` giảm đúng 9, `b` giảm 2 ⇒ bất biến giữ. Nếu Alice điền vào nửa trái (hoặc phải khi `a = b`) ⇒ Bob điền **cùng chữ số** vào nửa đối diện: `d` không đổi, `a` và `b` cùng giảm 1 ⇒ bất biến giữ. (Luôn còn `?` cho Bob đi vì tổng số `?` chẵn.) Cuối cùng `a = b = 0` ⇒ `d = 0` ⇒ Bob thắng.

**(3) `a + b` chẵn và `2d ≠ 9(b − a)` ⇒ Alice thắng.** Giả sử `2d > 9(b − a)`. Alice luôn điền `9` vào trái (nếu còn) hoặc `0` vào phải — mỗi nước của Alice làm `2d − 9(b − a)` **tăng hoặc giữ** ở mức có lợi, còn nước của Bob làm đại lượng này thay đổi trong khoảng `[−9, 9]` theo hướng ngược; ghép cặp (Alice, Bob) thì đại lượng `2d − 9(b − a)` không giảm ⇒ khi hết `?` vẫn `> 0` ⇒ `d ≠ 0`. Trường hợp `<` đối xứng. (Mục 7.3 và 12 kiểm chứng bằng máy.)

## 11. Phân tích Time / Space

Gọi `n = len(num)`, `q` = số `?`.

| Cách | Time | Space |
|---|---|---|
| 1. Minimax trên chuỗi | `O(11^q · 10q)` | `O(11^q)` |
| 2. Minimax nén `(a, b, d)` | `O(a · b · q · 20)` | như số trạng thái |
| 3. Công thức | `O(n)` | `O(1)` |

In [15]:
rng = random.Random(77)
big = "".join(rng.choice("0123456789?") for _ in range(10**5))
t = time.perf_counter(); r = Solution().sumGame(big); dt = time.perf_counter() - t
print(f"n = 10^5: {r} trong {1000*dt:.1f} ms")
t = time.perf_counter(); r2 = SolutionMemo().sumGame("?" * 30 + "5" * 30); dt2 = time.perf_counter() - t
print(f"minimax nén với 30 dấu ?: {r2} trong {1000*dt2:.0f} ms | công thức: {Solution().sumGame('?' * 30 + '5' * 30)}")

n = 10^5: True trong 14.2 ms
minimax nén với 30 dấu ?: True trong 3 ms | công thức: True


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu minimax)

In [16]:
sols = [Solution(), SolutionFormula(), SolutionMemo()]
edge = examples + [
    ("??", False),          # Bob bắt chước
    ("9?", True),           # 1 dấu ? -> Alice đi cuối
    ("0??0", False),
    ("?9", True),
    ("9??0", True),         # 2d = 18, 9(b-a) = 0
    ("90??", False),        # d = 9, b - a = 2 -> 2d = 18 = 9*2
    ("????", False),
    ("1??2", True),
]
for s, want in edge:
    assert sum_game_brute(s) == want, (s, sum_game_brute(s))
    for sol in sols:
        assert sol.sumGame(s) == want, (s, sol)
    print("OK", repr(s), "->", want)

OK '5023' -> False
OK '25??' -> True
OK '?3295???' -> False
OK '??' -> False
OK '9?' -> True
OK '0??0' -> False
OK '?9' -> True
OK '9??0' -> True
OK '90??' -> False
OK '????' -> False
OK '1??2' -> True


### 12.2 Vét cạn mọi chuỗi nhỏ (bảng chữ `{0, 5, 9, ?}`, độ dài ≤ 6)

In [17]:
total = 0
for L in (2, 4, 6):
    for t_ in product("059?", repeat=L):
        s = "".join(t_)
        if s.count("?") > 4:
            continue
        want = sum_game_brute(s)
        assert Solution().sumGame(s) == want and SolutionMemo().sumGame(s) == want, s
        total += 1
print("đã kiểm", total, "chuỗi — công thức khớp minimax trên chuỗi")

đã kiểm 4349 chuỗi — công thức khớp minimax trên chuỗi


### 12.3 Ngẫu nhiên: công thức vs minimax nén với nhiều `?` hơn

In [18]:
rng = random.Random(42)
for _ in range(500):
    h = rng.randint(1, 10)
    s = "".join(rng.choice("0123456789??") for _ in range(2 * h))
    if s.count("?") > 10:
        continue
    assert Solution().sumGame(s) == SolutionMemo().sumGame(s), s
print("500 chuỗi ngẫu nhiên: công thức khớp minimax nén")

500 chuỗi ngẫu nhiên: công thức khớp minimax nén


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng chia nguyên `d == 9*(b − a)//2` mà quên xét tính chẵn lẻ

In [19]:
def wrong_intdiv(num):
    a, sl, b, sr = stats(num)
    return (sl - sr) != 9 * (b - a) // 2                 # SAI khi a + b lẻ: // làm tròn

for s_ in ["4?", "?4", "13?5", "9?"]:
    print(repr(s_), "sai:", wrong_intdiv(s_), "| đúng:", Solution().sumGame(s_))

'4?' sai: False | đúng: True
'?4' sai: True | đúng: True
'13?5' sai: True | đúng: True
'9?' sai: True | đúng: True


**Vì sao sai:** với `"4?"`: `a = 0, b = 1, d = 4`, `9·1 // 2 = 4` ⇒ bản sai tưởng Bob cân bằng được. Thực tế chỉ có **một** `?` và Alice điền nó (nước cuối) ⇒ Alice thắng. Khi `a + b` lẻ thì `9(b − a)/2` **không nguyên**, nên phải xử lý riêng.

**Điều thú vị:** nếu viết bằng **nhân chéo** `2d != 9(b − a)` thì khi `a + b` lẻ, vế phải lẻ còn vế trái chẵn ⇒ luôn khác ⇒ tự động trả `True`. Tức là phép kiểm chẵn lẻ riêng là **thừa** với dạng nhân chéo — nhưng viết rõ ra vẫn giúp code dễ hiểu.

### Lỗi 2 — Đảo dấu hiệu (dùng `sR − sL`)

In [20]:
def wrong_sign(num):
    a, sl, b, sr = stats(num)
    if (a + b) % 2: return True
    return 2 * (sr - sl) != 9 * (b - a)                 # SAI: dấu

for s, want in examples + [("90??", False)]:
    print(repr(s), "sai:", wrong_sign(s), "| đúng:", Solution().sumGame(s))

'5023' sai: False | đúng: False
'25??' sai: True | đúng: True
'?3295???' sai: True | đúng: False
'90??' sai: True | đúng: False


**Vì sao sai:** nửa có nhiều `?` hơn (ở đây nửa phải, `b > a`) cần được **bù** đúng lượng `9·(b − a)/2` để bằng nửa kia ⇒ tổng **trái** phải lớn hơn: `sL − sR = 9(b − a)/2`. Đảo dấu là đòi ngược lại.

### Lỗi 3 — "Còn `?` là Alice thắng" / "Bob luôn bắt chước được"

In [21]:
def wrong_any_q(num):
    return "?" in num or stats(num)[1] != stats(num)[3]   # SAI

def wrong_always_copy(num):
    a, sl, b, sr = stats(num)
    return (a + b) % 2 == 1 or sl != sr                   # SAI: bỏ qua phần '?' dư ở một nửa

for s in ["??", "?3295???", "25??", "90??"]:
    print(repr(s), "còn ? là thắng:", wrong_any_q(s), "| chỉ bắt chước:", wrong_always_copy(s), "| đúng:", Solution().sumGame(s))

'??' còn ? là thắng: True | chỉ bắt chước: False | đúng: False
'?3295???' còn ? là thắng: True | chỉ bắt chước: True | đúng: False
'25??' còn ? là thắng: True | chỉ bắt chước: True | đúng: True
'90??' còn ? là thắng: True | chỉ bắt chước: True | đúng: False


**Vì sao sai:** `"??"` Bob thắng dù còn `?`. Còn bắt chước chỉ xử lý các cặp `?` **trái–phải**; phần `?` **dư** ở một nửa buộc Bob dùng chiến lược "bù 9", nên điều kiện là `d = 9·(b − a)/2` chứ không phải `d = 0`.

### Lỗi 4 — Dùng phép chia số thực `d == 9 * (b - a) / 2`

In [22]:
def wrong_float(num):
    a, sl, b, sr = stats(num)
    if (a + b) % 2: return True
    return (sl - sr) != 9 * (b - a) / 2                  # đúng về toán, nhưng phụ thuộc float

print([wrong_float(s) == Solution().sumGame(s) for s, _ in examples])

[True, True, True]


**Ghi chú:** với số nhỏ thì vẫn đúng, nhưng thói quen tốt là nhân chéo `2d == 9(b − a)` để tránh số thực và lỗi làm tròn với số lớn (và trong ngôn ngữ khác, `/` giữa số nguyên là chia nguyên — `9*(b−a)/2` có thể bị cắt nếu bạn quên kiểm tính chẵn lẻ trước).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Trò chơi điền số, mục tiêu "bằng nhau / khác nhau"** ⇒ nghĩ ngay chiến lược **bắt chước (mirror)** và **ghép cặp tổng hằng** (`x` và `9 − x`).
- **Ai đi nước cuối** thường quyết định khi nước cuối "tự do" ⇒ kiểm tính chẵn lẻ số lượt.
- Nén trạng thái theo **đại lượng bất biến** (hiệu, số lượng) để minimax chạy được, rồi đoán công thức từ bảng kết quả.
- So sánh phân số bằng nhân chéo.

## 15. Biến thể của bài

1. **Chữ số từ `0` đến `k − 1`** (thay vì 0–9) ⇒ Bob thắng ⇔ `q` chẵn và `2d = (k − 1)(b − a)` (Bài tập 5).
2. **Bob đi trước** ⇒ vai trò nước cuối đảo; phân tích lại tính chẵn lẻ.
3. **Mục tiêu "tổng trái lớn hơn tổng phải"** ⇒ mỗi người tối ưu một chiều — tham lam điền `9`/`0`.
4. **Nhiều nhóm thay vì hai nửa** ⇒ minimax nén theo vector hiệu; tổng quát khó hơn.
5. Họ bài "Nim / bắt chước": LeetCode 292, 1908, 877 (*Stone Game* — người đi trước luôn thắng nhờ chiến lược chẵn/lẻ).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax trên chuỗi | thử mọi vị trí × chữ số | `O(11^q)` | `O(11^q)` | kiểm chứng `q ≤ 4` |
| 2. Minimax nén | trạng thái `(a, b, d, lượt)` | đa thức theo `q` | như số trạng thái | đoán/kiểm công thức |
| 3. Công thức | chẵn lẻ + `2d = 9(b − a)` | `O(n)` | `O(1)` | **nộp bài** |
| 4. Mô phỏng chiến lược | cho chiến lược thật đấu ngẫu nhiên | — | — | thuyết phục chứng minh |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `"??"`, `"9?"`, `"90??"`, `"?9?9"`?

**Bài 2 (tính tay):** Tính `a, sL, b, sR` cho `"?3295???"` và áp công thức.

**Bài 3 (điền TODO):** Viết `bob_reply(a, b, d, last_move)` — nước đáp trả của Bob theo chiến lược ở mục 10 (giả sử đang ở trạng thái Bob thắng).

**Bài 4 (tự viết lại):** In bảng kết quả minimax nén cho `a, b ∈ [0, 3]`, `d ∈ [−20, 20]` và tự rút ra công thức (khi `a + b` chẵn, Bob thắng ở đúng các `d` nào?).

**Bài 5 (mini challenge):** Tổng quát hóa: chữ số chỉ từ `0` đến `k − 1` (`2 ≤ k ≤ 10`). Viết `sum_game_k(num, k)` và đối chiếu với minimax.

In [23]:
def bob_reply(a, b, d, last_move):
    # TODO
    pass

def table_bob_wins(a, b, drange=range(-20, 21)):
    # TODO: trả về list d mà Bob thắng (lượt Alice, a + b chẵn)
    pass

def sum_game_k(num, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Đếm `?` | `"9?"` lẻ ⇒ Alice | `"90??"`: `2·9 = 9·2` ⇒ Bob; `"?9?9"`: a=1,b=1,d=0 ⇒ Bob |
| 2 | Trái `?329`, phải `5???` | `a=1, sL=14, b=3, sR=5` | `18 = 18` ⇒ false |
| 3 | Alice đi ở nửa còn dư `?` ⇒ bù `9 − x` cùng nửa | Ngược lại bắt chước nửa đối diện | Dùng lại `bob_strategy` |
| 4 | Gọi `win_compressed(a, b, d, True)` | Lọc các `d` trả `False` | Thấy đúng một giá trị `d = 9(b − a)/2` |
| 5 | Thay `9` bằng `k − 1` | Minimax nén với chữ số `0..k−1` | `2d == (k − 1)(b − a)` |

## 19. Đáp án tham khảo

In [24]:
r1 = [Solution().sumGame(s) for s in ["??", "9?", "90??", "?9?9"]]
print("Bài 1:", r1); assert r1 == [False, True, False, False]
print("Bài 2:"); assert sum_game_formula_demo("?3295???") is False

Bài 1: [False, True, False, False]
Bài 2:
? trái a=1, tổng trái sL=14, ? phải b=3, tổng phải sR=5
2(sL - sR) = 18, 9(b - a) = 18 -> bằng: Bob thắng


In [25]:
def bob_reply(a, b, d, last_move):
    return bob_strategy(a, b, d, last_move)

def table_bob_wins(a, b, drange=range(-20, 21)):
    return [d for d in drange if not win_compressed(a, b, d, True)]

for a in range(4):
    for b in range(4):
        if (a + b) % 2 == 0:
            ws = table_bob_wins(a, b, range(-40, 41))
            assert ws == [9 * (b - a) // 2]
            print(f"a={a}, b={b}: Bob thắng đúng khi d ∈ {ws}")
rng = random.Random(4)
for _ in range(200):
    h = rng.randint(1, 5); s = "".join(rng.choice("0123456789??") for _ in range(2 * h))
    a, sl, b, sr = stats(s)
    if (a + b) % 2 == 0 and not Solution().sumGame(s):
        assert duel(s, random_player(rng), bob_reply) is False
print("Bài 3, 4: OK")

a=0, b=0: Bob thắng đúng khi d ∈ [0]
a=0, b=2: Bob thắng đúng khi d ∈ [9]
a=1, b=1: Bob thắng đúng khi d ∈ [0]
a=1, b=3: Bob thắng đúng khi d ∈ [9]
a=2, b=0: Bob thắng đúng khi d ∈ [-9]
a=2, b=2: Bob thắng đúng khi d ∈ [0]
a=3, b=1: Bob thắng đúng khi d ∈ [-9]
a=3, b=3: Bob thắng đúng khi d ∈ [0]
Bài 3, 4: OK


In [26]:
@lru_cache(maxsize=None)
def win_k(a, b, d, alice_turn, k):
    if a == 0 and b == 0:
        return d != 0
    moves = ([(a - 1, b, d + x) for x in range(k)] if a else []) + ([(a, b - 1, d - x) for x in range(k)] if b else [])
    res = (win_k(na, nb, nd, not alice_turn, k) for na, nb, nd in moves)
    return any(res) if alice_turn else all(res)

def sum_game_k(num, k):
    a, sl, b, sr = stats(num)
    return (a + b) % 2 == 1 or 2 * (sl - sr) != (k - 1) * (b - a)

rng = random.Random(9)
for _ in range(600):
    k = rng.randint(2, 10)
    h = rng.randint(1, 4)
    digits = "".join(str(x) for x in range(k))
    s = "".join(rng.choice(digits + "??") for _ in range(2 * h))
    a, sl, b, sr = stats(s)
    assert sum_game_k(s, k) == win_k(a, b, sl - sr, True, k), (s, k)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
h = len(num) // 2
a = num[:h].count('?'); b = num[h:].count('?')
d = sum(int(c) for c in num[:h] if c != '?') - sum(int(c) for c in num[h:] if c != '?')
if (a + b) % 2: return True                 # Alice đi nước cuối
return 2 * d != 9 * (b - a)                 # Bob thắng <=> d == 9(b - a)/2
```

**Checklist:**
- [ ] Tên hàm `sumGame`, trả `True` khi **Alice** thắng?
- [ ] Trường hợp tổng số `?` lẻ ⇒ `True`?
- [ ] Dấu: `d = sL − sR`, so với `9(b − a)/2` (nửa nhiều `?` hơn cần được bù)?
- [ ] Nhân chéo, không chia số thực?
- [ ] `O(n)`: đếm một lượt.